In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/My Drive/Crime Analysis/')
%ls

Mounted at /content/drive
bunseki_conpe/  data/  final_data/  tochijihai_hackathon/


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

# DataFrame Initializations

Common DataFrames of target labels and subregion polygons (with some basic features)

In [3]:
target_df = pd.read_csv('data/region/region_2024_total.csv').set_index('市区町丁', drop=True, inplace=False)
target_df = target_df.drop(['latitude', 'longitude'], axis=1)

def convert_kanji_to_num(s):
    if pd.isna(s):
        return s
    return s.replace('〇', '０').replace('一', '１').replace('二', '２').replace('三', '３').replace('四', '４').replace('五', '５').replace('六', '６').replace('七', '７').replace('八', '８').replace('九', '９')

subregions = gpd.read_file('data/subregions.geojson')
subregions = subregions.to_crs(epsg=4326)
subregions['市区町丁'] = subregions.apply(lambda row: f"{row['CITY_NAME']}{convert_kanji_to_num(row['S_NAME'])}", axis=1)
subregions = subregions.set_index('市区町丁', drop=True, inplace=False)
subregions = subregions.loc[~subregions.index.duplicated(keep='first')]

common_regions = list(set(target_df.index) & set(subregions.index))
common_regions = pd.Index(common_regions).unique()
target_df = target_df.loc[common_regions]
subregions = subregions.loc[common_regions]

In [4]:
def read_gpd(file_path, **kwargs):
    try:
        gdf = gpd.read_file(file_path, **kwargs)
        return gdf
    except Exception as e:
        print(f"Error reading the GeoJSON file: {e}")
        return None

Use the function below for any GDF with shapely points

In [5]:
def count_points_in_subregions(points, subregions, categorical_columns):
    """
    Count the number of points in each subregion based on categorical features.

    Parameters:
    - points (GeoDataFrame): A GeoDataFrame containing point geometries of points.
    - subregions (GeoDataFrame): A GeoDataFrame containing polygon geometries of subregions.
    - categorical_columns (list): A list of column names in 'points' to be used for categorical counting.

    Returns:
    - pd.DataFrame: A DataFrame with subregions as the index and counts of each category as columns.
    """
    if points.crs == None:
        points.set_crs(epsg=4326, inplace=True)
    if points.crs != subregions.crs:
        points = points.to_crs(subregions.crs)

    points_in_subregions = gpd.sjoin(points, subregions, how="inner", predicate="within")

    encoded_points = pd.DataFrame(index=points_in_subregions.index)
    for column in categorical_columns:
        dummies = pd.get_dummies(points_in_subregions[column], prefix=column)
        encoded_points = pd.concat([encoded_points, dummies], axis=1)
    encoded_points['subregion'] = points_in_subregions.index_right

    aggregated = encoded_points.groupby('subregion').sum()

    result = pd.DataFrame(index=subregions.index)
    result = result.join(aggregated, how='left').fillna(0).astype(int)

    return result

### Schools

In [ ]:
schools = read_gpd('bunseki_conpe/datasets/point_data/schools.geojson')
schools = schools.rename(columns={
    'P29_003': '学校分類',
    'P29_006': '管理者コード',
    'P29_007': '休校区分',
    'P29_008': 'キャンパスコード'
})

schools['学校分類'] = schools['学校分類'].astype(int).map({
    16001: "小学校",
    16002: "中学校",
    16003: "中等教育学校",
    16004: "高等学校",
    16005: "高等専門学校",
    16006: "短期大学",
    16007: "大学",
    16011: "幼稚園",
    16012: "特別支援学校",
    16013: "幼保連携型認定こども園",
    16014: "義務教育学校",
    16015: "各種学校",
    16016: "専修学校"
})
schools['管理者コード'] = schools['管理者コード'].astype(int).map({
    1: "国",
    2: "都道府県",
    3: "市区町村",
    4: "民間",
    0: "その他"
})
schools['休校区分'] = schools['休校区分'].astype(int).map({
    0: "調査なし",
    1: "開校中",
    2: "休校中"
})
schools['キャンパスコード'] = schools['キャンパスコード'].astype('category')

In [ ]:
schools_aggregate = count_points_in_subregions(schools, subregions, ['学校分類', '管理者コード', '休校区分']) # , 'キャンパスコード'

### Customer Facilities

In [21]:
facilities = read_gpd('bunseki_conpe/datasets/point_data/customers.shp', encoding='shift-jis')
facilities = facilities.rename(columns={
    'P33_004': '施設区分コード',
    'P33_014': '公民館の種別'
})

facilities['施設区分コード'] = facilities['施設区分コード'].astype(int).map({
    1: '映画館',
    2: '公会堂・集会場',
    3: '劇場・演劇場',
    4: '展示場',
    5: '寄席を有する体育館・観覧場',
    6: 'その他集客施設'
})
facilities['公民館の種別'] = facilities['公民館の種別'].astype(int).map({
    1: '中央',
    2: '地区',
    3: '分館',
    4: 'その他'
})

In [22]:
facilities_aggregate = count_points_in_subregions(facilities, subregions, ['施設区分コード', '公民館の種別'])

### Police Stations

In [ ]:
policestations = read_gpd('bunseki_conpe/datasets/point_data/policestations.shp', encoding='shift-jis')
policestations = policestations.rename(columns={
    'P18_003': '種別コード',
})

policestations['種別コード'] = policestations['種別コード'].astype(int).map({
    1: "警察本部",
    2: "警察署",
    3: "分庁舎",
    4: "交番",
    5: "駐在所",
    6: "派出所",
    7: "警察学校",
    8: "地域安全センター、連絡所等"
})

In [ ]:
policestations_aggregate = count_points_in_subregions(policestations, subregions, ['種別コード'])

## Correlation / Relation metrics

In [ ]:
# Pearson Correlation
combined_df = policestations_aggregate.join(target_df)
correlation_matrix = combined_df.corr(method='pearson')

target_correlations = correlation_matrix[target_df.columns]
target_correlations.to_csv('bunseki_conpe/correlation_matrix.csv', index=True)

In [23]:
# Mutual Information Regression
from sklearn.feature_selection import mutual_info_regression

mi_scores_df = pd.DataFrame()

for target in target_df.columns:
    y = target_df[target]

    mi = mutual_info_regression(facilities_aggregate, y)

    mi_scores = pd.DataFrame({'Feature': facilities_aggregate.columns, 'Mutual Information': mi})
    mi_scores['Target'] = target
    mi_scores_df = pd.concat([mi_scores_df, mi_scores], axis=0)

mi_scores_df = mi_scores_df.pivot(index='Target', columns='Feature', values='Mutual Information')
mi_scores_df.to_csv('bunseki_conpe/mi_scores.csv', index=True)

In [24]:
n_shuffles = 1000  # Takes about 4:30
shuffled_mi_scores = []

for i in range(n_shuffles):
    if i % 100 == 0:
        print(f"Shuffling iteration {i}/{n_shuffles}")
    shuffled_y = target_df[target].sample(frac=1, random_state=i).reset_index(drop=True)  # Shuffle target
    mi_shuffled = mutual_info_regression(facilities_aggregate, shuffled_y)
    shuffled_mi_scores.extend(mi_shuffled)

shuffled_mi_scores = pd.DataFrame(shuffled_mi_scores, columns=['Mutual Information'])
threshold = shuffled_mi_scores['Mutual Information'].quantile(0.95)

print(f"Threshold MI score based on shuffled data (95th percentile): {threshold}")

mi_scores_df = mi_scores_df.melt(var_name='Feature', value_name='Mutual Information', ignore_index=False).reset_index()
significant_mi_scores = mi_scores_df[mi_scores_df['Mutual Information'] > threshold]

print("Significant MI scores based on threshold:")
significant_mi_scores

Shuffling iteration 0/1000
Shuffling iteration 100/1000
Shuffling iteration 200/1000
Shuffling iteration 300/1000
Shuffling iteration 400/1000
Shuffling iteration 500/1000
Shuffling iteration 600/1000
Shuffling iteration 700/1000
Shuffling iteration 800/1000
Shuffling iteration 900/1000
Threshold MI score based on shuffled data (95th percentile): 0.018774915548637805
Significant MI scores based on threshold:


,Target,Feature,Mutual Information
54,凶悪犯計,公民館の種別_中央,0.019864
70,非侵入窃盗自販機ねらい,公民館の種別_中央,0.020105
102,非侵入窃盗オートバイ盗,公民館の種別_分館,0.019133
166,粗暴犯傷害,施設区分コード_その他集客施設,0.020872
179,非侵入窃盗置引き,施設区分コード_その他集客施設,0.020702
188,その他計,施設区分コード_公会堂・集会場,0.022897
210,非侵入窃盗すり,施設区分コード_公会堂・集会場,0.024725
211,非侵入窃盗その他,施設区分コード_公会堂・集会場,0.019741
219,非侵入窃盗自転車盗,施設区分コード_公会堂・集会場,0.025081
225,その他計,施設区分コード_劇場・演劇場,0.029078
